In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time
import re
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

os.makedirs('data', exist_ok=True)

BASE_URL = 'https://pakmag.net/film/details.php?pid={}'
MAX_PID = 4580
DELAY = 0.5
SAVE_EVERY = 100
OUTPUT_FILE = 'data/pakmag_films.csv'

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})
session.verify = False

In [ ]:
PAKISTANI_LANGUAGES = {'urdu', 'punjabi', 'pashto', 'sindhi', 'balochi',
                       'pushto', 'pushtu', 'pahari', 'saraiki'}


def get_field(table, label):
    if table is None:
        return None
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) >= 2 and label.lower() in cells[0].get_text(strip=True).lower():
            value = cells[1].get_text(separator=', ', strip=True)
            value = re.sub(r',\s*,', ',', value).strip(', ')
            return value if value and value != '?' else None
    return None


def parse_film_page(pid, html):
    soup = BeautifulSoup(html, 'html.parser')

    h1 = soup.find('h1')
    if h1 is None:
        return None
    title = h1.get_text(strip=True)
    if not title:
        return None

    year = None
    year_match = re.search(r'\((\d{4})\)', soup.get_text())
    if year_match:
        year = int(year_match.group(1))

    language = None
    for tag in soup.find_all(['h4', 'h5', 'h6', 'p', 'small']):
        text = tag.get_text(strip=True).lower()
        for lang in PAKISTANI_LANGUAGES:
            if lang in text and len(text) < 30:
                language = tag.get_text(strip=True)
                break
        if language:
            break

    page_title_tag = soup.find('title')
    if page_title_tag and not language:
        for lang in PAKISTANI_LANGUAGES:
            if lang in page_title_tag.get_text().lower():
                language = lang.capitalize()
                break

    full_text = soup.get_text().lower()
    is_pakistani = 'pakistani' in full_text or (language and language.lower() in PAKISTANI_LANGUAGES)

    credits_table = None
    for table in soup.find_all('table'):
        table_text = table.get_text().lower()
        if 'actor' in table_text or 'director' in table_text:
            credits_table = table
            break

    actors = get_field(credits_table, 'actor')
    director = get_field(credits_table, 'director')
    producer = get_field(credits_table, 'producer')
    writer = get_field(credits_table, 'writer')
    music_director = get_field(credits_table, 'musician')

    genre = None
    genre_match = re.search(r'Genre:\s*([^\n<]+)', soup.get_text())
    if genre_match:
        genre = genre_match.group(1).strip()

    imdb_id = None
    imdb_match = re.search(r'(tt\d{7,8})', str(soup))
    if imdb_match:
        imdb_id = imdb_match.group(1)

    return {
        'pid': pid,
        'title': title,
        'year': year,
        'language': language,
        'genre': genre,
        'director': director,
        'actors': actors,
        'producer': producer,
        'writer': writer,
        'music_director': music_director,
        'imdb_id': imdb_id,
        'is_pakistani': is_pakistani,
        'source_url': BASE_URL.format(pid),
    }

In [ ]:
test_results = []
for pid in [1, 100, 500, 1000, 4580]:
    try:
        r = session.get(BASE_URL.format(pid), timeout=10)
        result = parse_film_page(pid, r.text)
        if result:
            test_results.append(result)
    except Exception:
        pass
    time.sleep(DELAY)

pd.DataFrame(test_results)

In [ ]:
if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    scraped_pids = set(existing['pid'].tolist())
    records = existing.to_dict('records')
else:
    scraped_pids = set()
    records = []

for pid in range(1, MAX_PID + 1):
    if pid in scraped_pids:
        continue
    try:
        r = session.get(BASE_URL.format(pid), timeout=15)
        film = parse_film_page(pid, r.text)
        if film:
            records.append(film)
    except requests.exceptions.RequestException:
        pass

    if pid % SAVE_EVERY == 0:
        pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)

    time.sleep(DELAY)

pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)

In [ ]:
raw_df = pd.read_csv(OUTPUT_FILE)

str_cols = raw_df.select_dtypes(include='object').columns
raw_df[str_cols] = raw_df[str_cols].apply(lambda c: c.str.strip())
raw_df['year'] = pd.to_numeric(raw_df['year'], errors='coerce')

pakistani_df = raw_df[raw_df['is_pakistani'] == True].copy()
pakistani_df = pakistani_df.drop_duplicates(subset=['pid'])

pakistani_df = pakistani_df[['pid', 'title', 'year', 'language', 'genre',
                             'director', 'actors', 'producer', 'writer',
                             'music_director', 'imdb_id', 'is_pakistani', 'source_url']]

pakistani_df.to_csv('data/pakmag_pakistani_films_clean.csv', index=False)
pakistani_df.head()